In [214]:
# characterize the IPA enriched upstream regulators and causal network regulators from the significantly dysregulated proteomics data
## identify the signifcantly enriched regulators across all timepoint and compile the unique/shared target genes per timepoint
import pandas as pd
import numpy as np
import os

In [215]:
# get paths for all IPA exported files
paths = os.listdir('../data/IPA_export/upstream')
# paths

['1hr_PHOS_causal.txt',
 '2hr_PHOS.txt',
 '24hr_PROT.txt',
 '47hr_PROT_causal.txt',
 '24hr_PHOS.txt',
 '2hr_PROT_regeffects.txt',
 '24hr_PROT_causal.txt',
 '2hr_PHOS_causal.txt',
 '47hr_PROT_regeffects.txt',
 '2hr_PROT.txt',
 '1hr_PROT_causal.txt',
 '47hr_PROT.txt',
 '1hr_PROT.txt',
 '24hr_PROT_regeffects.txt',
 '47hr_PHOS_causal.txt',
 '24hr_PHOS_regeffects.txt',
 '24hr_PHOS_causal.txt',
 '1hr_PHOS.txt',
 '1hr_PROT_regeffects.txt',
 '47hr_PHOS.txt',
 '2hr_PROT_causal.txt']

In [ ]:
# store respective paths/datatype in dictionary

pathd['prot'] = {'upstream': [],
                'causal': [],
                'regeffects': []}
# pathd['phos'] = {'upstream': [],
#                 'causal': [],
#                 'regeffects': []}

In [ ]:
# read in data as df in dictionary
for p in paths:
    if "PROT" in p:
        df = pd.read_csv('../data/IPA_export/upstream/'+p,sep='\t',header=1)
        df['sample'] = p[:-4]
        
        if "causal" in p:
            pathd['prot']['causal'].append(df)
        elif 'regeffects' in p:
            pathd['prot']['regeffects'].append(df)
            
        else:
            pathd['prot']['upstream'].append(df)
            
    # elif "PHOS" in p:
    #     df = pd.read_csv('../data/IPA_export/upstream/'+p,sep='\t',header=1)
    #     df['sample'] = p[:-4]
        
    #     if "causal" in p:
    #         pathd['phos']['causal'].append(df)
    #     elif 'regeffects' in p:
    #         pathd['phos']['regeffects'].append(df)
            
    #     else:
    #         pathd['phos']['upstream'].append(df)
# pathd

{'prot': {'upstream': [                                Upstream Regulator Expr Log Ratio  \
   0                                             IRGM                  
   1                                 Interferon alpha                  
   2                                            RC3H1                  
   3                                            IFNL1                  
   4                                              PRL                  
   ..                                             ...            ...   
   435                                           MAFB                  
   436  miR-146a-5p (and other miRNAs w/seed GAGAACU)                  
   437                                           KLF5                  
   438                                      MACROH2A1          0.084   
   439                                          KDM1A          0.053   
   
                  Molecule Type Predicted Activation State Activation z-score  \
   0                     enzyme

In [ ]:
# init another dictionary to store filtered data
filtered={}
filtered['prot'] = {'upstream': [],
                'causal': []}
# filtered['phos'] = {'upstream': [],
#                 'causal': []}

In [220]:
# created "filtered" significant data tables into filtered dict
for p in [*pathd]:
    for p1 in ['upstream','causal']:
        for df in pathd[p][p1]:
            for index, row in df.iterrows():
                z = row['Activation z-score']
                pv = row['p-value of overlap']
                sample = row['sample']
                
                if z != ' ' and pv != ' ':
                    if abs(float(z))>=2 and float(pv) <= 0.05:
                        df.loc[index,'sig'] = True
            
            if 'sig' in df.columns:
                print(p+p1+sample)
                sigdf = df[df['sig'] == True]
                filtered[p][p1].append(sigdf)

protupstream24hr_PROT
protupstream2hr_PROT
protupstream47hr_PROT
protupstream1hr_PROT
protcausal47hr_PROT_causal
protcausal24hr_PROT_causal
protcausal1hr_PROT_causal
protcausal2hr_PROT_causal
phosupstream24hr_PHOS
phoscausal1hr_PHOS_causal
phoscausal24hr_PHOS_causal


In [221]:
# get unique set of genes per sample in upstream datatables
upstream_inter = {}
for df in filtered['prot']['upstream']:
    sample = df['sample'].iloc[0]
    setgenes = set(df['Upstream Regulator'])
    upstream_inter[sample] = setgenes

In [222]:
# get unique set of genes per sample in causal datatables
causal_inter = {}
for df in filtered['prot']['causal']:
    sample = df['sample'].iloc[0]
    csetgenes = set(df['Master Regulator'])
    causal_inter[sample] = csetgenes

In [223]:
# upstream regulators that are significant in all timepoints/datasets (1; 'MYC')
# set.intersection(upstream_inter['1hr_PROT'],upstream_inter['2hr_PROT'],upstream_inter['47hr_PROT'],upstream_inter['24hr_PROT'])

{'MYC'}

In [224]:
# causal network regulators that are significant in all timepoints/datasets (7; {'AREG', 'ATM', 'CUL4B', 'MYC', 'Pkg', 'ROCK', 'Rac'})
## these regulators are of interest in downstream analysis
# set.intersection(causal_inter['1hr_PROT_causal'],causal_inter['2hr_PROT_causal'],causal_inter['47hr_PROT_causal'],causal_inter['24hr_PROT_causal'])

{'AREG', 'ATM', 'CUL4B', 'MYC', 'Pkg', 'ROCK', 'Rac'}

In [225]:

for i in [*upstream_inter]:
    print(i)
    print(len(upstream_inter[i]))
    
for i in [*causal_inter]:
    print(i)
    print(len(causal_inter[i]))    

24hr_PROT
65
2hr_PROT
32
47hr_PROT
32
1hr_PROT
19
47hr_PROT_causal
107
24hr_PROT_causal
166
1hr_PROT_causal
121
2hr_PROT_causal
156


In [226]:
# get unique upstreamreg genes per timepoint
u1 = list(set(upstream_inter['1hr_PROT'])-set(upstream_inter['2hr_PROT']))
u12 = list(set(u1)-set(upstream_inter['47hr_PROT']))
u13 = list(set(u12)-set(upstream_inter['24hr_PROT']))
upstream_inter['1hr_PROT_unique'] = u13
print(len(u13))

u2 = list(set(upstream_inter['2hr_PROT'])-set(upstream_inter['1hr_PROT']))
u22 = list(set(u2)-set(upstream_inter['47hr_PROT']))
u23 = list(set(u22)-set(upstream_inter['24hr_PROT']))
upstream_inter['2hr_PROT_unique'] = u23
print(len(u23))

u3 = list(set(upstream_inter['47hr_PROT'])-set(upstream_inter['1hr_PROT']))
u32 = list(set(u3)-set(upstream_inter['2hr_PROT']))
u33 = list(set(u32)-set(upstream_inter['24hr_PROT']))
upstream_inter['47hr_PROT_unique'] = u33
print(len(u33))

u4 = list(set(upstream_inter['24hr_PROT'])-set(upstream_inter['1hr_PROT']))
u42 = list(set(u4)-set(upstream_inter['2hr_PROT']))
u43 = list(set(u42)-set(upstream_inter['47hr_PROT']))
upstream_inter['24hr_PROT_unique'] = u43
print(len(u43))

9
9
6
54


In [227]:
# get unique upstreamreg genes per timepoint
c1 = list(set(causal_inter['1hr_PROT_causal'])-set(causal_inter['2hr_PROT_causal']))
c12 = list(set(c1)-set(causal_inter['47hr_PROT_causal']))
c13 = list(set(c12)-set(causal_inter['24hr_PROT_causal']))
causal_inter['1hr_PROT_causal_unique'] = c13
print(len(c13))

c2 = list(set(causal_inter['2hr_PROT_causal'])-set(causal_inter['1hr_PROT_causal']))
c22 = list(set(c2)-set(causal_inter['47hr_PROT_causal']))
c23 = list(set(c22)-set(causal_inter['24hr_PROT_causal']))
causal_inter['2hr_PROT_causal_unique'] = c23
print(len(c23))

c3 = list(set(causal_inter['47hr_PROT_causal'])-set(causal_inter['1hr_PROT_causal']))
c32 = list(set(c3)-set(causal_inter['2hr_PROT_causal']))
c33 = list(set(c32)-set(causal_inter['24hr_PROT_causal']))
causal_inter['47hr_PROT_causal_unique'] = c33
print(len(c33))

c4 = list(set(causal_inter['24hr_PROT_causal'])-set(causal_inter['1hr_PROT_causal']))
c42 = list(set(c4)-set(causal_inter['2hr_PROT_causal']))
c43 = list(set(c42)-set(causal_inter['47hr_PROT_causal']))
causal_inter['24hr_PROT_causal_unique'] = c43
print(len(c43))

59
67
27
125


In [228]:
for df in filtered['prot']['upstream']:
    sample = df['sample'].iloc[0]
    print(sample,df.shape)
    uniqgenes = list(upstream_inter[sample+'_unique'])
    print('uniqgenes',len(uniqgenes))
    df_uniq = df[df['Upstream Regulator'].isin(uniqgenes)]
    df_uniq['sample'] = sample+'_unique'
    print(df_uniq.shape)
    filtered['prot']['upstream'].append(df_uniq)


24hr_PROT (65, 10)
uniqgenes 54
(54, 10)
2hr_PROT (32, 10)
uniqgenes 9
(9, 10)
47hr_PROT (32, 10)
uniqgenes 6
(6, 10)
1hr_PROT (19, 11)
uniqgenes 9
(9, 11)
24hr_PROT_unique (54, 10)


<ipython-input-228-56e4a48779a2>:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_uniq['sample'] = sample+'_unique'


KeyError: '24hr_PROT_unique_unique'

In [229]:
for df in filtered['prot']['causal']:
    sample = df['sample'].iloc[0]
    print(sample,df.shape)
    cuniqgenes = list(causal_inter[sample+'_unique'])
    print('uniqgenes',len(cuniqgenes))
    df_uniqc = df[df['Master Regulator'].isin(cuniqgenes)]
    df_uniqc['sample'] = sample+'_unique'
    print(df_uniqc.shape)
    filtered['prot']['causal'].append(df_uniqc)

47hr_PROT_causal (122, 14)
uniqgenes 27
(9, 11)
24hr_PROT_causal (190, 14)
uniqgenes 125
(9, 11)
1hr_PROT_causal (125, 15)
uniqgenes 59
(9, 11)
2hr_PROT_causal (178, 14)
uniqgenes 67
(9, 11)
47hr_PROT_causal_unique (27, 14)


<ipython-input-229-87aa397060af>:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_uniqc['sample'] = sample+'_unique'


KeyError: '47hr_PROT_causal_unique_unique'

In [198]:
# write csv for the significant filtered and unique upstream regulators
for df in filtered['prot']['upstream']:
    sample = df['sample'].iloc[0]
    print(sample)

    if "unique" not in sample:
        df.to_csv('../results/IPA_downstream_analysis/upstreamreg_sig/' + sample + '_sig.csv')
    else:
        df.to_csv('../results/IPA_downstream_analysis/upstreamreg_sig/' + sample+'.csv')

24hr_PROT
2hr_PROT
47hr_PROT
1hr_PROT
24hr_PROT_unique
2hr_PROT_unique
47hr_PROT_unique
1hr_PROT_unique


In [232]:
for df in filtered['prot']['causal']:
    sample = df['sample'].iloc[0][:-7]
    print(sample,df['sample'].iloc[0])

47hr_PROT 47hr_PROT_causal
24hr_PROT 24hr_PROT_causal
1hr_PROT 1hr_PROT_causal
2hr_PROT 2hr_PROT_causal
47hr_PROT_causal 47hr_PROT_causal_unique
24hr_PROT_causal 24hr_PROT_causal_unique
1hr_PROT_causal 1hr_PROT_causal_unique
2hr_PROT_causal 2hr_PROT_causal_unique


In [251]:
# find and filter for genes that are both in upstream reg_sig and causal
for df in filtered['prot']['causal']:
    if 'overlap_upstreamreg' in df.columns:
        break
    sample = df['sample'].iloc[0][:-7]
    print(sample)
    
    if 'causal' in sample:
        sample = sample[:-7]
    
    #extract causal 'master regulator' genes
    causgenes = set(df['Master Regulator'])
    
    print(len(causgenes))
    
    #extract corresponding sig upstream reg genes
    upstr_genes = list(upstream_inter[sample]) # upstream genes matching timepoint
        
    # find if any genes in common and write into orig df as new columns
    for index, row in df.iterrows():
        
        gene = row['Master Regulator']
        if gene in upstr_genes:
            df.loc[index,'overlap_upstreamreg'] = True
        
    print(sum(df['overlap_upstreamreg']))  
        


47hr_PROT
107
nan
24hr_PROT
166
nan
1hr_PROT
121
nan
2hr_PROT
156
nan
47hr_PROT_causal
27
nan
24hr_PROT_causal
125
nan
1hr_PROT_causal
59
nan
2hr_PROT_causal
67
nan


/Users/jasminetat/opt/anaconda3/lib/python3.8/site-packages/pandas/core/indexing.py:1596: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.obj[key] = _infer_fill_value(value)
/Users/jasminetat/opt/anaconda3/lib/python3.8/site-packages/pandas/core/indexing.py:1765: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  isetter(loc, value)


In [261]:
# write csv for the significant filtered and unique causal regulators
for df in filtered['prot']['causal']:
    sample = df['sample'].iloc[0]
    print(sample,df.shape)

    if "unique" in sample:
        df.to_csv('../results/IPA_downstream_analysis/upstream_causal/' + sample+'_unique.csv')
    else:
        df.to_csv('../results/IPA_downstream_analysis/upstream_causal/' + sample+'_sig.csv')

47hr_PROT_causal (122, 15)
24hr_PROT_causal (190, 15)
1hr_PROT_causal (125, 16)
2hr_PROT_causal (178, 15)
47hr_PROT_causal_unique (27, 15)
24hr_PROT_causal_unique (143, 15)
1hr_PROT_causal_unique (61, 16)
2hr_PROT_causal_unique (72, 15)


In [7]:
regs_of_interest = set.intersection(causal_inter['1hr_PROT_causal'],causal_inter['2hr_PROT_causal'],causal_inter['47hr_PROT_causal'],causal_inter['24hr_PROT_causal'])
regs_of_interest

{'AREG', 'ATM', 'CUL4B', 'MYC', 'Pkg', 'ROCK', 'Rac'}

In [6]:
regs_of_interest = {'AREG', 'ATM', 'CUL4B', 'MYC', 'Pkg', 'ROCK', 'Rac'}

In [2]:
import os
# os.listdir('../results/IPA_downstream_analysis/upstream_causal/')

['24hr_PROT_causal_sig.csv',
 '1hr_PROT_causal_unique_unique.csv',
 '47hr_PROT_causal_unique_unique.csv',
 '.DS_Store',
 '1hr_PROT_causal_sig.csv',
 '2hr_PROT_causal_unique_unique.csv',
 '24hr_PROT_causal_unique_unique.csv',
 '47hr_PROT_causal_sig.csv',
 '2hr_PROT_causal_sig.csv']

In [39]:
# create a "shared" datatable of the sig. regulators in common for all timepoints

upstream_casual_df = {} # datatables into dict
shared_append = [] # init to store filtered tables of each df

for df_name in os.listdir('../results/IPA_downstream_analysis/upstream_causal/'):
    if "csv" in df_name:
        print(df_name)
        upstream_casual_df[df_name] = pd.read_csv('../results/IPA_downstream_analysis/upstream_causal/' + df_name)
        
        i_df = upstream_casual_df[df_name][upstream_casual_df[df_name]['Master Regulator'].isin(list(regs_of_interest))]

        if '_sig' in df_name:
            shared_append.append(i_df)
            
            
# combine all data and filter for regulators of interest
shared_df = pd.concat(shared_append).drop_duplicates()
shared_df['abs_z-score'] = shared_df['Activation z-score'].abs()
shared_df = shared_df.sort_values(['Master Regulator','sample','abs_z-score'])
shared_df = shared_df.drop_duplicates(subset=['Master Regulator','sample'],keep='last')

## drop intermediate/unnecessary cols
shared_df = shared_df.drop(['Unnamed: 0','abs_z-score'],axis=1)
shared_df

24hr_PROT_causal_sig.csv
1hr_PROT_causal_unique_unique.csv
47hr_PROT_causal_unique_unique.csv
1hr_PROT_causal_sig.csv
2hr_PROT_causal_unique_unique.csv
24hr_PROT_causal_unique_unique.csv
47hr_PROT_causal_sig.csv
2hr_PROT_causal_sig.csv


,Master Regulator,Expr Log Ratio,Molecule Type,Participating regulators,Depth,Predicted Activation,Activation z-score,p-value of overlap,Network bias-corrected p-value,Target Molecules in Dataset,Causal network,Target-connected regulators,sample,sig,overlap_upstreamreg,Notes
2,AREG,,growth factor,"Akt,AKT1,Ap1,AREG,CEBPA,Creb,EGFR,ERBB3,ERK1/2...",3,Inhibited,-3.395,4.790000e-09,0.0033,"ACO1,AKR1B1,AKT1,ANO10,ARPC5L,CDKN2C,CP,DAB2,E...",68 (31),30,1hr_PROT_causal,True,NaN,biased
51,AREG,,growth factor,"AREG,EGFR,ERK1/2,MAPK1",2,Inhibited,-4.041,7.150000e-15,0.0001,"ARRB1,AURKA,B2M,BSG,C3,CCL4,CD44,CEBPA,CHI3L1,...",75 (4),4,24hr_PROT_causal,True,NaN,NaN
78,AREG,,growth factor,"Akt,AKT1,Ap1,AREG,CASP9,CEBPA,CHEK1,Creb,CTNNB...",3,Inhibited,-3.549,1.370000e-13,0.0220,"A2M,ACO1,ADAM10,ADAR,AGO2,AKR1B1,AKT1,ALB,ALDO...",223 (46),46,2hr_PROT_causal,True,NaN,NaN
38,AREG,,growth factor,"Akt,AKT1,Ap1,AREG,CEBPA,Creb,CTNNB1,DUSP1,EGFR...",3,Inhibited,-3.245,4.210000e-13,0.0411,"A2M,ABCB1,ACLY,ACO1,ADAM10,AKR1B1,AKT1,ALB,ALD...",247 (43),43,47hr_PROT_causal,True,NaN,NaN
5,ATM,0.160,kinase,"AKT1,AR,ATM,CD247,CEBPA,CHEK1,Creb,E2F3,EGFR,E...",3,Inhibited,-3.286,7.180000e-09,0.0087,"ADH5,ADK,AK1,AKR1B1,AKT1,AKT2,ANO10,ARPC5L,ATA...",89 (41),36,1hr_PROT_causal,True,NaN,
69,ATM,0.539,kinase,"ATM,CHEK1,CREB1,DNA-PK,E2f,ELF4,ERK1/2,FANCD2,...",2,Activated,3.012,7.650000e-14,0.0160,"ABCC1,AK1,ARRB1,B2M,BAX,BID,BUB1B,C3,CASP1,CBX...",135 (20),20,24hr_PROT_causal,True,NaN,NaN
24,ATM,0.247,kinase,"AKT1,AR,ATM,CASP8,CD247,CDK2,CEBPA,CHEK1,CHEK2...",3,Inhibited,-3.242,3.380000e-17,0.0067,"A2M,ABCE1,ACSL3,ACTR3,ADAM10,ADAR,ADGRE2,ADH5,...",320 (64),59,2hr_PROT_causal,True,NaN,NaN
9,ATM,0.519,kinase,"AKT1,AR,ATM,CASP8,CEBPA,CHEK1,CHEK2,Creb,CREB1...",3,Inhibited,-2.650,1.030000e-16,0.0170,"A2M,ABCC1,ABCE1,ACSL3,ACTR3,ADAM10,ADH5,ADK,AH...",356 (61),56,47hr_PROT_causal,True,NaN,NaN
84,CUL4B,-0.269,other,"CUL4B,MYC",2,Activated,3.300,2.150000e-06,0.0005,"AKT1,BAG1,EIF4E,GLO1,IFIT2,IQGAP2,MYD88,NME1,P...",18 (2),2,1hr_PROT_causal,True,NaN,biased
129,CUL4B,0.351,other,"CUL4B,MYC",2,Activated,4.323,2.300000e-10,0.0001,"ABCC1,ALCAM,BAG1,BAX,BCOR,BZW2,CD44,COL1A1,COX...",45 (2),2,24hr_PROT_causal,True,True,NaN


In [47]:
# consolidate all the causal regulators unique/sig per timepoint and write into excel

writer = pd.ExcelWriter('../results/IPA_downstream_analysis/PROT_causalreg.xlsx', engine='openpyxl')


# add shared df first
shared_df.to_excel(writer, sheet_name='shared')

for df_name in os.listdir('../results/IPA_downstream_analysis/upstream_causal/'):
    df = pd.read_csv('../results/IPA_downstream_analysis/upstream_causal/' + df_name)
    df.to_excel(writer, sheet_name=df_name.replace('_unique','',1))
    
writer.save()